In [ ]:
import unittest
import numpy as np
import pandas as pd
import joblib
import json
import os
import matplotlib
# استخدام backend غير تفاعلي لمنع تعطل النواة
matplotlib.use('Agg')  # Important: Add this before importing plt
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# =============================================================================
# ModelComparator Class 
# =============================================================================
class ModelComparator:
    def __init__(self, X_train, X_test, y_train, y_test):
        self.X_train = X_train
        self.X_test = X_test
        self.y_train = y_train
        self.y_test = y_test
        self.results = {}
        self.models = {}
        
    def train_random_forest(self, n_estimators=100, max_depth=20):
        """Train Random Forest model"""
        print("Training Random Forest...")
        start_time = datetime.now()
        
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=42,
            n_jobs=-1,
            class_weight='balanced'
        )
        
        model.fit(self.X_train, self.y_train)
        training_time = (datetime.now() - start_time).total_seconds()
        
        y_pred = model.predict(self.X_test)
        accuracy = accuracy_score(self.y_test, y_pred)
        precision = precision_score(self.y_test, y_pred, average='weighted', zero_division=0)
        recall = recall_score(self.y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(self.y_test, y_pred, average='weighted', zero_division=0)
        
        self.models['Random Forest'] = model
        self.results['Random Forest'] = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'training_time': training_time
        }
        
        print(f"Random Forest - Accuracy: {accuracy:.4f}, Time: {training_time:.2f}s")
        return model
    
    def train_gradient_boosting(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        """Train Gradient Boosting model"""
        print("Training Gradient Boosting...")
        start_time = datetime.now()
        
        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            random_state=42
        )
        
        model.fit(self.X_train, self.y_train)
        training_time = (datetime.now() - start_time).total_seconds()
        
        y_pred = model.predict(self.X_test)
        accuracy = accuracy_score(self.y_test, y_pred)
        precision = precision_score(self.y_test, y_pred, average='weighted', zero_division=0)
        recall = recall_score(self.y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(self.y_test, y_pred, average='weighted', zero_division=0)
        
        self.models['Gradient Boosting'] = model
        self.results['Gradient Boosting'] = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'training_time': training_time
        }
        
        print(f"Gradient Boosting - Accuracy: {accuracy:.4f}, Time: {training_time:.2f}s")
        return model
    
    def train_svm(self, C=1.0, kernel='rbf'):
        """Train SVM model"""
        print("Training SVM...")
        start_time = datetime.now()
        
        model = SVC(
            C=C,
            kernel=kernel,
            probability=True,
            random_state=42,
            class_weight='balanced'
        )
        
        model.fit(self.X_train, self.y_train)
        training_time = (datetime.now() - start_time).total_seconds()
        
        y_pred = model.predict(self.X_test)
        accuracy = accuracy_score(self.y_test, y_pred)
        precision = precision_score(self.y_test, y_pred, average='weighted', zero_division=0)
        recall = recall_score(self.y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(self.y_test, y_pred, average='weighted', zero_division=0)
        
        self.models['SVM'] = model
        self.results['SVM'] = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'training_time': training_time
        }
        
        print(f"SVM - Accuracy: {accuracy:.4f}, Time: {training_time:.2f}s")
        return model
    
    def compare_models(self):
        """Compare all trained models"""
        comparison_df = pd.DataFrame(self.results).T
        comparison_df = comparison_df.sort_values('accuracy', ascending=False)
        return comparison_df
    
    def get_best_model(self):
        """Get the best model based on accuracy"""
        best_model_name = max(self.results.items(), key=lambda x: x[1]['accuracy'])[0]
        best_model = self.models[best_model_name]
        best_accuracy = self.results[best_model_name]['accuracy']
        return best_model_name, best_model, best_accuracy

# =============================================================================
# Test Class
# =============================================================================
class TestModelComparator(unittest.TestCase):
    
    def setUp(self):
        """Set up test data for each test"""
        np.random.seed(42)
        n_samples = 200
        
        # Create test data
        self.X = np.random.randn(n_samples, 5)
        self.y = np.random.choice([0, 1], size=n_samples)
        
        # Split and scale data
        X_train, X_test, y_train, y_test = train_test_split(
            self.X, self.y, test_size=0.2, random_state=42
        )
        
        scaler = StandardScaler()
        self.X_train_scaled = scaler.fit_transform(X_train)
        self.X_test_scaled = scaler.transform(X_test)
        self.y_train = y_train
        self.y_test = y_test
    
    def test_random_forest_training(self):
        """Test Random Forest model training"""
        print("Testing Random Forest...")
        
        comparator = ModelComparator(self.X_train_scaled, self.X_test_scaled, self.y_train, self.y_test)
        model = comparator.train_random_forest(n_estimators=10, max_depth=5)
        
        self.assertIsNotNone(model)
        self.assertIn('Random Forest', comparator.models)
        self.assertIn('accuracy', comparator.results['Random Forest'])
        
        accuracy = comparator.results['Random Forest']['accuracy']
        self.assertGreaterEqual(accuracy, 0)
        self.assertLessEqual(accuracy, 1)
    
    def test_gradient_boosting_training(self):
        """Test Gradient Boosting model training"""
        print("Testing Gradient Boosting...")
        
        comparator = ModelComparator(self.X_train_scaled, self.X_test_scaled, self.y_train, self.y_test)
        model = comparator.train_gradient_boosting(n_estimators=10, max_depth=3)
        
        self.assertIsNotNone(model)
        self.assertIn('Gradient Boosting', comparator.models)
        self.assertIn('accuracy', comparator.results['Gradient Boosting'])
    
    def test_svm_training(self):
        """Test SVM model training"""
        print("Testing SVM...")
        
        comparator = ModelComparator(self.X_train_scaled, self.X_test_scaled, self.y_train, self.y_test)
        model = comparator.train_svm(C=1.0)
        
        self.assertIsNotNone(model)
        self.assertIn('SVM', comparator.models)
        self.assertIn('accuracy', comparator.results['SVM'])
    
    def test_model_comparison(self):
        """Test model comparison functionality"""
        print("Testing model comparison...")
        
        comparator = ModelComparator(self.X_train_scaled, self.X_test_scaled, self.y_train, self.y_test)
        
        # Train all models
        comparator.train_random_forest(n_estimators=10, max_depth=5)
        comparator.train_gradient_boosting(n_estimators=10, max_depth=3)
        comparator.train_svm(C=1.0)
        
        # Compare models
        comparison_df = comparator.compare_models()
        
        self.assertIsInstance(comparison_df, pd.DataFrame)
        self.assertEqual(len(comparison_df), 3)
        self.assertIn('accuracy', comparison_df.columns)
    
    def test_best_model_selection(self):
        """Test best model selection"""
        print("Testing best model selection...")
        
        comparator = ModelComparator(self.X_train_scaled, self.X_test_scaled, self.y_train, self.y_test)
        
        # Train models
        comparator.train_random_forest(n_estimators=10, max_depth=5)
        comparator.train_gradient_boosting(n_estimators=10, max_depth=3)
        
        # Get best model
        best_name, best_model, best_accuracy = comparator.get_best_model()
        
        self.assertIsNotNone(best_name)
        self.assertIsNotNone(best_model)
        self.assertGreaterEqual(best_accuracy, 0)
        self.assertLessEqual(best_accuracy, 1)
        self.assertIn(best_name, ['Random Forest', 'Gradient Boosting'])

# =============================================================================
# Main Test Runner
# =============================================================================
def run_classical_tests():
    """Run all classical model tests"""
    print("=" * 60)
    print("CLASSICAL MODEL COMPARISON TESTS")
    print("=" * 60)
    
    # Create test suite
    loader = unittest.TestLoader()
    suite = loader.loadTestsFromTestCase(TestModelComparator)
    
    # Run tests
    runner = unittest.TextTestRunner(verbosity=2)
    result = runner.run(suite)
    
    # Print summary
    print("=" * 60)
    print("TEST SUMMARY")
    print("=" * 60)
    print(f"Tests run: {result.testsRun}")
    print(f"Errors: {len(result.errors)}")
    print(f"Failures: {len(result.failures)}")
    print(f"Successful: {result.testsRun - len(result.errors) - len(result.failures)}")
    
    return result.wasSuccessful()

def demo_model_comparison():
    """Demonstrate the model comparison functionality"""
    print("\n" + "=" * 60)
    print("DEMO: MODEL COMPARISON")
    print("=" * 60)
    
    # Create demo data
    np.random.seed(42)
    X = np.random.randn(1000, 5)
    y = np.random.choice([0, 1], size=1000, p=[0.6, 0.4])
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    comparator = ModelComparator(X_train_scaled, X_test_scaled, y_train, y_test)
    
    print("Training models...")
    comparator.train_random_forest(n_estimators=20, max_depth=5)
    comparator.train_gradient_boosting(n_estimators=20, max_depth=3)
    comparator.train_svm(C=1.0)
    
    print("\nModel Comparison Results:")
    comparison_df = comparator.compare_models()
    
    for model_name in comparison_df.index:
        acc = comparison_df.loc[model_name, 'accuracy']
        prec = comparison_df.loc[model_name, 'precision']
        rec = comparison_df.loc[model_name, 'recall']
        f1 = comparison_df.loc[model_name, 'f1']
        time = comparison_df.loc[model_name, 'training_time']
        
        print(f"{model_name:.<20} Accuracy: {acc:.3f} | Precision: {prec:.3f} | "
              f"Recall: {rec:.3f} | F1: {f1:.3f} | Time: {time:.2f}s")
    
    # Get best model
    best_name, best_model, best_accuracy = comparator.get_best_model()
    print(f"\n Best Model: {best_name} with accuracy: {best_accuracy:.4f}")
    
    return comparison_df

if __name__ == '__main__':
    # Run tests
    print("Running classical model tests...")
    test_success = run_classical_tests()
    
    # Run demo only if tests passed
    if test_success:
        print("\n" + "=" * 60)
        print("STARTING DEMONSTRATION")
        print("=" * 60)
        
        # Run demo with error handling
        try:
            results_df = demo_model_comparison()
            print("\n Demonstration completed successfully!")
            
            os.makedirs('results', exist_ok=True)
            results_df.to_csv('results/model_comparison_results.csv')
            print(" Results saved to 'results/model_comparison_results.csv'")
            
        except Exception as e:
            print(f" Error in demonstration: {e}")
            print("  But tests passed successfully!")
    
    # Exit with appropriate code
    exit(0 if test_success else 1)

test_best_model_selection (__main__.TestModelComparator)
Test best model selection ... 

Running classical model tests...
CLASSICAL MODEL COMPARISON TESTS
Testing best model selection...
Training Random Forest...


ok
test_gradient_boosting_training (__main__.TestModelComparator)
Test Gradient Boosting model training ... 

Random Forest - Accuracy: 0.5750, Time: 0.15s
Training Gradient Boosting...
Gradient Boosting - Accuracy: 0.5750, Time: 0.13s
Testing Gradient Boosting...
Training Gradient Boosting...


ok
test_model_comparison (__main__.TestModelComparator)
Test model comparison functionality ... 

...

ok
test_random_forest_training (__main__.TestModelComparator)
Test Random Forest model training ... 

SVM - Accuracy: 0.4500, Time: 0.11s
Testing Random Forest...
Training Random Forest...


ok
test_svm_training (__main__.TestModelComparator)
Test SVM model training ... ok

----------------------------------------------------------------------
Ran 5 tests in 2.086s

OK


Random Forest - Accuracy: 0.5750, Time: 0.49s
Testing SVM...
Training SVM...
SVM - Accuracy: 0.4500, Time: 0.09s
TEST SUMMARY
Tests run: 5
Errors: 0
Failures: 0
Successful: 5

STARTING DEMONSTRATION

DEMO: MODEL COMPARISON
Training models...
Training Random Forest...
Random Forest - Accuracy: 0.5400, Time: 0.35s
Training Gradient Boosting...
Gradient Boosting - Accuracy: 0.6450, Time: 0.30s
Training SVM...
SVM - Accuracy: 0.5450, Time: 0.94s

Model Comparison Results:
Gradient Boosting... Accuracy: 0.645 | Precision: 0.669 | Recall: 0.645 | F1: 0.535 | Time: 0.30s
SVM................. Accuracy: 0.545 | Precision: 0.577 | Recall: 0.545 | F1: 0.553 | Time: 0.94s
Random Forest....... Accuracy: 0.540 | Precision: 0.545 | Recall: 0.540 | F1: 0.542 | Time: 0.35s

 Best Model: Gradient Boosting with accuracy: 0.6450

 Demonstration completed successfully!
 Results saved to 'results/model_comparison_results.csv'


: 